# 03 — Exploratory Data Analysis
Before building any model, we explore the data to understand patterns, spot issues, and form hypotheses about what drives Mets attendance.

Unlike the Padres (who sell out nearly every game), the Mets have significant attendance variance — ranging from ~15k to 42k. This makes for a more interesting and informative model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

TEAM_ID = 121  # New York Mets
SEASONS = [2022, 2023, 2024, 2025]

df = pd.read_csv("../data/mets_master.csv")
df["date"] = pd.to_datetime(df["date"])
df["attendance"] = df["attendance"].astype(int)

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

## Data Quality Check

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
print("Attendance summary:")
print(df["attendance"].describe().round(0))

## Attendance Distribution
How spread out is attendance? Is it normally distributed or skewed?

In [ ]:
CAPACITY = 41922  # Citi Field official capacity

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df["attendance"], bins=30, color="#002D72", edgecolor="white")
axes[0].axvline(df["attendance"].mean(), color="orange", linestyle="--", label=f"Mean: {df['attendance'].mean():,.0f}")
axes[0].axvline(CAPACITY, color="red", linestyle="--", label=f"Capacity: {CAPACITY:,}")
axes[0].set_xlabel("Attendance")
axes[0].set_ylabel("Number of Games")
axes[0].set_title("Attendance Distribution (2022-2025)")
axes[0].legend()

axes[1].boxplot(df["attendance"], patch_artist=True, boxprops=dict(facecolor="#002D72"))
axes[1].set_ylabel("Attendance")
axes[1].set_title("Attendance Boxplot")
axes[1].set_xticks([])

plt.tight_layout()
plt.show()

## Attendance by Season

In [ ]:
season_stats = df.groupby("season")["attendance"].agg(["mean", "median", "min", "max"]).round(0).astype(int)
print(season_stats)

season_means = df.groupby("season")["attendance"].mean()
plt.bar(season_means.index, season_means.values, color="#002D72", edgecolor="white")
plt.axhline(CAPACITY, color="red", linestyle="--", label=f"Capacity: {CAPACITY:,}")
plt.xlabel("Season")
plt.ylabel("Average Attendance")
plt.title("Average Attendance by Season")
plt.legend()
plt.show()

## Attendance by Day of Week

In [ ]:
df["day_of_week"] = df["date"].dt.day_name()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

day_avg = df.groupby("day_of_week")["attendance"].mean().reindex(day_order)

colors = ["#002D72" if d not in ["Saturday", "Sunday"] else "#FF6600" for d in day_order]
plt.bar(day_avg.index, day_avg.values, color=colors, edgecolor="white")
plt.xlabel("Day of Week")
plt.ylabel("Average Attendance")
plt.title("Average Attendance by Day of Week")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print(day_avg.round(0).astype(int))

## Attendance by Month
New York weather makes month a much stronger signal than for San Diego — cold April games and hot summer games are meaningfully different.

In [ ]:
df["month"] = df["date"].dt.month
month_names = {3: "Mar", 4: "Apr", 5: "May", 6: "Jun", 7: "Jul", 8: "Aug", 9: "Sep", 10: "Oct"}

month_avg = df.groupby("month")["attendance"].mean()
month_avg.index = month_avg.index.map(month_names)

plt.plot(month_avg.index, month_avg.values, marker="o", color="#002D72", linewidth=2)
plt.axhline(CAPACITY, color="red", linestyle="--", alpha=0.5, label="Capacity")
plt.xlabel("Month")
plt.ylabel("Average Attendance")
plt.title("Average Attendance by Month (2022-2025)")
plt.legend()
plt.tight_layout()
plt.show()

## Top and Bottom Opponents by Attendance

In [ ]:
opponent_avg = df.groupby("opponent")["attendance"].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

top10 = opponent_avg.head(10)
axes[0].barh(top10.index[::-1], top10.values[::-1], color="#002D72")
axes[0].set_title("Top 10 Opponents by Avg Attendance")
axes[0].set_xlabel("Average Attendance")

bot10 = opponent_avg.tail(10)
axes[1].barh(bot10.index[::-1], bot10.values[::-1], color="#888")
axes[1].set_title("Bottom 10 Opponents by Avg Attendance")
axes[1].set_xlabel("Average Attendance")

plt.tight_layout()
plt.show()

## Weather vs. Attendance
Unlike San Diego, New York weather is a real variable — cold April/May games and rain should show meaningful effects here.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

weather = df.dropna(subset=["avg_temp_f"])
axes[0].scatter(weather["avg_temp_f"], weather["attendance"], alpha=0.4, color="#002D72")
z = np.polyfit(weather["avg_temp_f"], weather["attendance"], 1)
p = np.poly1d(z)
x_line = np.linspace(weather["avg_temp_f"].min(), weather["avg_temp_f"].max(), 100)
axes[0].plot(x_line, p(x_line), color="orange", linewidth=2)
axes[0].set_xlabel("Avg Temp (F) at Game Time")
axes[0].set_ylabel("Attendance")
axes[0].set_title("Temperature vs. Attendance")
corr_temp = weather[["avg_temp_f", "attendance"]].corr().iloc[0, 1]
axes[0].text(0.05, 0.95, f"r = {corr_temp:.2f}", transform=axes[0].transAxes, fontsize=11)

axes[1].scatter(df["avg_precip_mm"], df["attendance"], alpha=0.4, color="#002D72")
axes[1].set_xlabel("Avg Precipitation (mm) at Game Time")
axes[1].set_ylabel("Attendance")
axes[1].set_title("Precipitation vs. Attendance")
corr_precip = df[["avg_precip_mm", "attendance"]].corr().iloc[0, 1]
axes[1].text(0.05, 0.95, f"r = {corr_precip:.2f}", transform=axes[1].transAxes, fontsize=11)

plt.tight_layout()
plt.show()

## Correlation Summary

In [ ]:
df["dayofweek"] = df["date"].dt.dayofweek
df["is_weekend"] = df["dayofweek"].isin([4, 5, 6]).astype(int)

numeric_cols = ["attendance", "season", "month", "dayofweek", "is_weekend", "avg_temp_f", "avg_precip_mm"]
corr = df[numeric_cols].corr()[["attendance"]].drop("attendance").sort_values("attendance", ascending=False)

print("Correlation with attendance:")
print(corr.round(3))

colors = ["#002D72" if v > 0 else "#888" for v in corr["attendance"].values[::-1]]
plt.barh(corr.index[::-1], corr["attendance"].values[::-1], color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Correlation with Attendance")
plt.title("Feature Correlations with Attendance")
plt.tight_layout()
plt.show()

## Game Time: Day Games vs. Night Games

In [ ]:
def get_game_times(team_id, seasons):
    times = {}
    for season in seasons:
        url = "https://statsapi.mlb.com/api/v1/schedule"
        params = {"sportId": 1, "season": season, "gameType": "R",
                  "teamId": team_id, "hydrate": "linescore,team"}
        data = requests.get(url, params=params).json()
        for date in data.get("dates", []):
            for game in date.get("games", []):
                if game["teams"]["home"]["team"]["id"] == team_id:
                    game_dt_str = game.get("gameDate", "")
                    if game_dt_str:
                        dt = pd.to_datetime(game_dt_str, utc=True)
                        # Convert UTC to Eastern (UTC-4 during baseball season / EDT)
                        hour_et = (dt.hour - 4) % 24
                        minute_et = dt.minute
                        times[game["gamePk"]] = hour_et + minute_et / 60
    return times

print("Fetching game times...")
game_times = get_game_times(TEAM_ID, SEASONS)
df["game_hour_et"] = df["game_pk"].map(game_times)
df["is_day_game"] = (df["game_hour_et"] < 17).astype(int)

print(f"Day games:   {df['is_day_game'].sum()}")
print(f"Night games: {(df['is_day_game'] == 0).sum()}")
print()
print("Most common start hours (ET):")
print(df["game_hour_et"].round(1).value_counts().sort_index())

In [ ]:
def time_bucket(hour):
    if hour < 17:
        return "Early (<5pm)"
    elif hour < 18.5:
        return "Twilight (5-6:30pm)"
    else:
        return "Night (6:30pm+)"

df["time_bucket"] = df["game_hour_et"].apply(time_bucket)

print("Games per time bucket:")
print(df["time_bucket"].value_counts())
print()
print("Average attendance per time bucket:")
print(df.groupby("time_bucket")["attendance"].mean().round(0).astype(int))

In [ ]:
# Weekday-only breakdown by time bucket
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
wd = df[df["day_of_week"].isin(weekdays)].copy()

bucket_order = ["Early (<5pm)", "Twilight (5-6:30pm)", "Night (6:30pm+)"]
pivot = wd.groupby(["day_of_week", "time_bucket"])["attendance"].mean().unstack()
pivot = pivot.reindex(index=weekdays, columns=bucket_order)

print("Weekday average attendance by time bucket:")
print(pivot.round(0).astype("Int64"))
print()

x = np.arange(len(weekdays))
width = 0.25
colors = ["#FF6600", "#5B9BD5", "#002D72"]
fig, ax = plt.subplots(figsize=(11, 5))
for i, (bucket, color) in enumerate(zip(bucket_order, colors)):
    vals = pivot[bucket].fillna(0)
    ax.bar(x + (i - 1) * width, vals, width, label=bucket, color=color)
ax.set_xticks(x)
ax.set_xticklabels(weekdays)
ax.set_ylabel("Average Attendance")
ax.set_title("Weekday Attendance by Game Time Bucket")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Full day/time breakdown including weekends
all_days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
bucket_order = ["Early (<5pm)", "Twilight (5-6:30pm)", "Night (6:30pm+)"]

avg_pivot = df.groupby(["day_of_week", "time_bucket"])["attendance"].mean().unstack().reindex(index=all_days, columns=bucket_order)
count_pivot = df.groupby(["day_of_week", "time_bucket"])["attendance"].count().unstack().reindex(index=all_days, columns=bucket_order)

print("Average attendance (n=game count):")
for day in all_days:
    row_parts = []
    for bucket in bucket_order:
        avg = avg_pivot.loc[day, bucket]
        cnt = count_pivot.loc[day, bucket]
        if pd.isna(avg):
            row_parts.append(f"{bucket[:6]}: ---")
        else:
            flag = " *" if cnt < 5 else ""
            row_parts.append(f"{bucket[:6]}: {int(avg):,} (n={int(cnt)}){flag}")
    print(f"{day:<12} | " + " | ".join(row_parts))

print()
print("* = fewer than 5 games, treat with caution")

## National Broadcasts

In [ ]:
def get_national_broadcasts(team_id, seasons):
    national = {}
    national_networks = {}
    for season in seasons:
        url = "https://statsapi.mlb.com/api/v1/schedule"
        params = {"sportId": 1, "season": season, "gameType": "R",
                  "teamId": team_id, "hydrate": "broadcasts"}
        data = requests.get(url, params=params).json()
        for date in data.get("dates", []):
            for game in date.get("games", []):
                if game["teams"]["home"]["team"]["id"] == team_id:
                    pk = game["gamePk"]
                    tv_national = [b["name"] for b in game.get("broadcasts", [])
                                   if b.get("type") == "TV" and b.get("isNational")]
                    national[pk] = 1 if tv_national else 0
                    national_networks[pk] = ", ".join(tv_national) if tv_national else None
    return national, national_networks

nat_flags, nat_networks = get_national_broadcasts(TEAM_ID, SEASONS)
df["is_national"] = df["game_pk"].map(nat_flags).fillna(0).astype(int)
df["national_network"] = df["game_pk"].map(nat_networks)

premium_networks = {"FOX", "FOX, FOX", "ESPN/ESPN App", "ESPN/ESPN App, ESPN/ESPN App",
                    "Apple TV", "Apple TV, Apple TV", "TBS (out-of-market only)",
                    "TBS (out-of-market only), TBS (out-of-market only)",
                    "TBS (out-of-market only), TBS", "Roku", "Roku, Roku",
                    "FS1", "FS1, FS1"}
df["is_premium_national"] = df["national_network"].isin(premium_networks).astype(int)

print(f"National broadcast games: {df['is_national'].sum()} / {len(df)}")
print()
print("By network:")
print(df[df["national_network"].notna()]["national_network"].value_counts())
print()
print("Average attendance by broadcast tier:")
tier_map = {(0, 0): "Non-national", (1, 0): "MLBN only", (1, 1): "Premium national"}
df["broadcast_tier"] = list(zip(df["is_national"], df["is_premium_national"]))
df["broadcast_tier"] = df["broadcast_tier"].map(tier_map)
print(df.groupby("broadcast_tier")["attendance"].mean().round(0).astype(int))